# Exploratory Notebook: Advanced Feature Engineering Pipeline

> **Note:** This is an *exploratory* notebook. The final model results are in `01_MAIN_ufc_fight_prediction.ipynb`.

This notebook explores an alternative, more granular feature engineering pipeline built on top of [Polars](https://pola.rs/) for high-performance data manipulation. Rather than the simple per-fight aggregation used in the main notebook, this approach:

1. **Unnests** the raw parquet into three relational tables: fights, prior fights, and prior rounds.
2. **Builds feature groups** independently — each function targets a specific domain of fighter behavior.
3. **Includes mirror augmentation** to correct a dataset label bias.

Many of these features were ultimately not used in the final model (due to the added complexity and marginal gain), but they represent meaningful signals worth exploring for future work.

---
**Feature groups covered:**
- Fight-level biometrics and differentials
- Duration & activity pace
- Win/loss method breakdown (KO, sub, decision)
- 3-round vs. 5-round format experience
- Striking stats across three temporal windows (last fight, last 3, career) + trend
- Advanced combat ratios (KD rate, control %, body/head/leg distribution)
- Round-level pacing (early vs. late), post-KD response
- Win/loss streak tracking
- Accumulated damage
- Fight tendency (finisher vs. decision fighter)
- Exponential recency-weighted striking stats
- Mirror augmentation to fix f1/f2 label bias

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path('data/fight_snapshots.parquet')

## 1. Data Loading and Unnesting

The raw parquet stores each fight's prior history as nested structs (`prior_f1`, `prior_f2`). We explode these into flat relational tables. This unlocks per-fight and per-round granularity that the main notebook's `process()` function cannot access.

In [ ]:
def get_raw_df() -> pl.DataFrame:
    """
    Loads the raw fight snapshot parquet from disk.
    Args:
        None
    Returns:
        pl.DataFrame with one row per root fight, including nested prior fight history
    """
    return pl.read_parquet(DATA_PATH)


def unnest_raw_df(df: pl.DataFrame) -> tuple:
    """
    Splits the nested fight snapshot DataFrame into three flat relational tables.
    Args:
        df: raw fight snapshot DataFrame with nested prior_f1 and prior_f2 columns
    Returns:
        tuple of (fights_df, prior_fights_df, prior_rounds_df) as Polars DataFrames
    """
    df = df.with_columns(pl.col('fight_id').alias('root_fight_id'))

    fights_df = df.drop(['prior_f1', 'prior_f2', 'fight_id'])

    def extract_prior_fights(df, col, role):
        """
        Explodes a nested prior-fights column into one row per prior fight.
        Args:
            df: raw DataFrame
            col: column name ('prior_f1' or 'prior_f2') to explode
            role: fighter role label ('f1' or 'f2') to tag each row
        Returns:
            pl.DataFrame with one row per prior fight, tagged with fighter_role
        """
        return (
            df.select(['root_fight_id', col])
            .explode(col)
            .unnest(col)
            .rename({'fight_id': 'prior_fight_id'})
            .with_columns(pl.lit(role).alias('fighter_role'))
            .drop('rounds')
        )

    prior_fights_df = pl.concat([
        extract_prior_fights(df, 'prior_f1', 'f1'),
        extract_prior_fights(df, 'prior_f2', 'f2'),
    ])

    def extract_prior_rounds(df, col, role):
        """
        Explodes round-level data nested within prior fights into one row per round.
        Args:
            df: raw DataFrame
            col: column name to explode ('prior_f1' or 'prior_f2')
            role: fighter role label to tag each row
        Returns:
            pl.DataFrame with one row per round, including fighter_role and round stats
        """
        return (
            df.select(['root_fight_id', col])
            .explode(col)
            .unnest(col)
            .rename({'fight_id': 'prior_fight_id'})
            .with_columns(pl.lit(role).alias('fighter_role'))
            .select(['root_fight_id', 'prior_fight_id', 'fighter_role', 'rounds'])
            .explode('rounds')
            .unnest('rounds')
        )

    prior_rounds_df = pl.concat([
        extract_prior_rounds(df, 'prior_f1', 'f1'),
        extract_prior_rounds(df, 'prior_f2', 'f2'),
    ])

    return fights_df, prior_fights_df, prior_rounds_df


raw = get_raw_df()
fights_df, prior_fights_df, prior_rounds_df = unnest_raw_df(raw)

print('fights_df:       ', fights_df.shape)
print('prior_fights_df: ', prior_fights_df.shape)
print('prior_rounds_df: ', prior_rounds_df.shape)

## 2. Fight-Level Features

These come directly from the root fight record: physical differentials (height, reach, age), stance matchup, weight class, and win-rate differential.

In [ ]:
FIGHT_TYPE_MAP = {'Title': 1, 'Main Event': 2, 'Co-Main Event': 3, 'Prelim': 4, 'Early Prelim': 5}
WEIGHT_CLASS_MAP = {
    'strawweight': 1, 'flyweight': 2, 'bantamweight': 3, 'featherweight': 4,
    'lightweight': 5, 'welterweight': 6, 'middleweight': 7,
    'light heavyweight': 8, 'heavyweight': 9, 'catch weight': 10,
}
STANCE_MAP = {'Orthodox': 1, 'Southpaw': 2, 'Switch': 3, 'Open Stance': 4}


def build_fight_features(fights: pl.DataFrame) -> pl.DataFrame:
    """
    Extracts fight-level predictive features from the root fights DataFrame.
    Computes physical differentials (height, reach, age), encoded categoricals
    (stance, weight class, fight type), win-rate differential, and southpaw advantage.
    Args:
        fights: pl.DataFrame with one row per root fight including fighter bios and records
    Returns:
        pl.DataFrame of fight-level features keyed on root_fight_id
    """
    return fights.select([
        pl.col('root_fight_id').alias('meta_root_fight_id'),
        pl.col('winner_id').alias('meta_winner_id'),
        pl.col('fighter1_id').alias('meta_f1_id'),
        pl.col('fighter2_id').alias('meta_f2_id'),
        pl.col('fight_date').alias('meta_fight_date'),
        pl.col('fight_format').cast(pl.Int16),
        pl.col('fight_type').replace(FIGHT_TYPE_MAP).cast(pl.Int8).alias('fight_type_id'),
        pl.col('weight_class').replace(WEIGHT_CLASS_MAP).cast(pl.Int8).alias('weight_class_id'),
        pl.col('prior_cnt_f1').alias('f1_fight_count'),
        pl.col('prior_cnt_f2').alias('f2_fight_count'),
        pl.col('f1_stance').replace(STANCE_MAP).cast(pl.Int8).alias('f1_stance_id'),
        pl.col('f2_stance').replace(STANCE_MAP).cast(pl.Int8).alias('f2_stance_id'),
        (pl.col('f1_height_in') - pl.col('f2_height_in')).alias('height_diff'),
        (pl.col('f1_reach_in') - pl.col('f2_reach_in')).alias('reach_diff'),
        ((pl.col('fight_date') - pl.col('f1_dob')).dt.total_days() / 365.25).cast(pl.Float32).alias('f1_age'),
        ((pl.col('fight_date') - pl.col('f2_dob')).dt.total_days() / 365.25).cast(pl.Float32).alias('f2_age'),
        ((pl.col('f2_dob') - pl.col('f1_dob')).dt.total_days() / 365.25).cast(pl.Float32).alias('age_diff'),
        (
            pl.col('f1_win').cast(pl.Float32) / (pl.col('f1_win') + pl.col('f1_loss')).cast(pl.Float32)
            - pl.col('f2_win').cast(pl.Float32) / (pl.col('f2_win') + pl.col('f2_loss')).cast(pl.Float32)
        ).alias('win_rate_diff'),
        pl.when((pl.col('f1_stance') == 'Southpaw') & (pl.col('f2_stance') == 'Orthodox')).then(pl.lit(1))
        .when((pl.col('f1_stance') == 'Orthodox') & (pl.col('f2_stance') == 'Southpaw')).then(pl.lit(-1))
        .otherwise(pl.lit(0)).cast(pl.Int8).alias('southpaw_advantage'),
    ])


fight_features = build_fight_features(fights_df)
print('Fight features shape:', fight_features.shape)
fight_features.head(3)

## 3. Duration & Activity Features

How long does this fighter typically fight, and how active have they been recently? These signal conditioning, ring rust, and pace of competition.

In [ ]:
def build_duration_features(prior_fights: pl.DataFrame) -> pl.DataFrame:
    """
    Computes fight-duration statistics from prior fight history.
    Captures how long fights last on average, trend over last 3, and total minutes fought.
    Args:
        prior_fights: pl.DataFrame with one row per prior fight including end_time (MM:SS string)
    Returns:
        pl.DataFrame grouped by (root_fight_id, fighter_role) with duration aggregates in seconds
    """
    end_time_s = (
        pl.col('end_time').str.split(':').list.get(0).cast(pl.Int32) * 60
        + pl.col('end_time').str.split(':').list.get(1).cast(pl.Int32)
    )
    pf = prior_fights.with_columns(end_time_s.alias('end_time_s'))
    return (
        pf.group_by(['root_fight_id', 'fighter_role'])
        .agg([
            pl.col('end_time_s').sort_by('fight_date', descending=True).first().alias('last_fight_end_time_s'),
            pl.col('end_time_s').sort_by('fight_date', descending=True).head(3).mean().alias('last_3_avg_end_time_s'),
            pl.col('end_time_s').mean().alias('avg_end_time_s'),
            pl.col('end_time_s').sum().alias('total_time_fought_s'),
        ])
    )


def build_years_since_last_fight(prior_fights: pl.DataFrame, root_dates: pl.DataFrame) -> pl.DataFrame:
    """
    Computes how many years have elapsed since each fighter's last fight before the root fight.
    A large value indicates ring rust or injury layoff.
    Args:
        prior_fights: pl.DataFrame with prior fight history including fight_date
        root_dates: pl.DataFrame with (root_fight_id, root_fight_date) for join
    Returns:
        pl.DataFrame with (root_fight_id, fighter_role, years_since_last_fight)
    """
    return (
        prior_fights.group_by(['root_fight_id', 'fighter_role'])
        .agg(pl.col('fight_date').max().alias('last_fight_date'))
        .join(root_dates, on='root_fight_id')
        .with_columns(
            ((pl.col('root_fight_date') - pl.col('last_fight_date')).dt.total_days() / 365.25)
            .cast(pl.Float32).alias('years_since_last_fight')
        )
        .select(['root_fight_id', 'fighter_role', 'years_since_last_fight'])
    )


def build_activity(prior_fights: pl.DataFrame, root_dates: pl.DataFrame) -> pl.DataFrame:
    """
    Computes activity pace metrics: average fights per year, fights in last 12 months,
    and fights in last 3 years. Signals how busy and active a fighter has been.
    Args:
        prior_fights: pl.DataFrame with one row per prior fight
        root_dates: pl.DataFrame with (root_fight_id, root_fight_date)
    Returns:
        pl.DataFrame grouped by (root_fight_id, fighter_role) with activity metrics
    """
    return (
        prior_fights.join(root_dates, on='root_fight_id')
        .group_by(['root_fight_id', 'fighter_role'])
        .agg([
            pl.col('fight_date').min().alias('first_fight_date'),
            pl.col('fight_date').count().alias('total_fights'),
            pl.col('root_fight_date').first(),
            pl.col('fight_date').filter(
                pl.col('fight_date') >= (pl.col('root_fight_date') - pl.duration(days=365))
            ).count().alias('fights_this_year'),
            pl.col('fight_date').filter(
                pl.col('fight_date') >= (pl.col('root_fight_date') - pl.duration(days=1095))
            ).count().alias('fights_last_3yrs'),
        ])
        .with_columns(
            (pl.col('total_fights').cast(pl.Float32) /
             ((pl.col('root_fight_date') - pl.col('first_fight_date')).dt.total_days() / 365.25))
            .alias('avg_fights_per_year')
        )
        .select(['root_fight_id', 'fighter_role', 'avg_fights_per_year', 'fights_this_year', 'fights_last_3yrs'])
    )


root_dates = fights_df.select(['root_fight_id', pl.col('fight_date').alias('root_fight_date')])
duration = build_duration_features(prior_fights_df)
activity = build_activity(prior_fights_df, root_dates)
layoff   = build_years_since_last_fight(prior_fights_df, root_dates)

print('Duration features:', duration.shape)
print('Activity features:', activity.shape)
duration.head(3)

## 4. Win/Loss Method Breakdown

How does this fighter win and lose? KO rate, submission rate, and decision rate tell us about finishing ability and chin durability under pressure.

In [ ]:
def build_method_counts(prior_fights: pl.DataFrame) -> pl.DataFrame:
    """
    Computes career KO/submission/decision win and loss rates from prior fight history.
    Args:
        prior_fights: pl.DataFrame with one row per prior fight including result and method columns
    Returns:
        pl.DataFrame grouped by (root_fight_id, fighter_role) with ko_win_rate, sub_win_rate,
        ko_loss_rate, and sub_loss_rate columns
    """
    win  = pl.col('result') == 'win'
    loss = pl.col('result') == 'loss'
    ko   = pl.col('method') == 'kotko'
    sub  = pl.col('method') == 'sub'
    dec  = pl.col('method').is_in(['d_unan', 'd_maj', 'd_split'])

    return (
        prior_fights.group_by(['root_fight_id', 'fighter_role'])
        .agg([
            (win & ko).sum().alias('ko_wins'),
            (win & sub).sum().alias('sub_wins'),
            (win & dec).sum().alias('dec_wins'),
            (loss & ko).sum().alias('ko_losses'),
            (loss & sub).sum().alias('sub_losses'),
            win.sum().alias('total_wins'),
            loss.sum().alias('total_losses'),
        ])
        .with_columns([
            (pl.col('ko_wins').cast(pl.Float32)  / pl.col('total_wins').cast(pl.Float32)).fill_nan(0.0).alias('ko_win_rate'),
            (pl.col('sub_wins').cast(pl.Float32) / pl.col('total_wins').cast(pl.Float32)).fill_nan(0.0).alias('sub_win_rate'),
            (pl.col('ko_losses').cast(pl.Float32)  / pl.col('total_losses').cast(pl.Float32)).fill_nan(0.0).alias('ko_loss_rate'),
            (pl.col('sub_losses').cast(pl.Float32) / pl.col('total_losses').cast(pl.Float32)).fill_nan(0.0).alias('sub_loss_rate'),
        ])
        .drop(['ko_wins', 'sub_wins', 'dec_wins', 'ko_losses', 'sub_losses', 'total_wins', 'total_losses'])
    )


def build_format_experience(prior_fights: pl.DataFrame) -> pl.DataFrame:
    """
    Counts how many 3-round and 5-round fights each fighter has had, split by win/loss.
    Helps identify fighters who thrive or struggle in championship (5-round) formats.
    Args:
        prior_fights: pl.DataFrame with fight_format (3 or 5) and result columns
    Returns:
        pl.DataFrame grouped by (root_fight_id, fighter_role) with round-format win/loss counts
    """
    win  = pl.col('result') == 'win'
    loss = pl.col('result') == 'loss'
    r3   = pl.col('fight_format') == 3
    r5   = pl.col('fight_format') == 5

    return (
        prior_fights.group_by(['root_fight_id', 'fighter_role'])
        .agg([
            (r3 & win).sum().alias('3rd_wins'),
            (r3 & loss).sum().alias('3rd_losses'),
            r5.sum().alias('5rd_fights'),
            (r5 & win).sum().alias('5rd_wins'),
            (r5 & loss).sum().alias('5rd_losses'),
        ])
    )


methods  = build_method_counts(prior_fights_df)
fmt_exp  = build_format_experience(prior_fights_df)
print('Method features:', methods.shape)
print('Format features:', fmt_exp.shape)
methods.head(3)

## 5. Striking Statistics (3 Temporal Windows + Trend)

One of the more sophisticated feature groups. Rather than just career averages, we compute stats over three windows — last fight, last 3 fights, and career — and compute a **trend ratio** (last_3 / career) to detect improving or declining fighters.

In [ ]:
def build_striking_stats(prior_fights: pl.DataFrame) -> pl.DataFrame:
    """
    Computes striking/grappling rates across three temporal windows and a trend signal.
    Stats computed: slpm (sig strikes per min), str_acc, sapm (absorbed), str_def,
    td_avg (takedowns per 15 min), td_acc, td_def, sub_avg.
    Three windows: last fight (lf_), last 3 fights (l3_), career (no prefix).
    Trend = last_3 / career; >1 means improving, <1 means declining; clipped to [0, 3].
    Args:
        prior_fights: pl.DataFrame with one row per prior fight including striking columns
    Returns:
        pl.DataFrame grouped by (root_fight_id, fighter_role) with 32 striking feature columns
    """
    end_time_s = (
        pl.col('end_time').str.split(':').list.get(0).cast(pl.Float32) * 60
        + pl.col('end_time').str.split(':').list.get(1).cast(pl.Float32)
    )
    pf = prior_fights.with_columns(end_time_s.alias('end_time_s'))

    agg_cols = ['sig_str_landed', 'sig_str_attempts', 'opp_sig_str_landed', 'opp_sig_str_attempts',
                'td_landed', 'td_attempts', 'opp_td_landed', 'opp_td_attempts', 'sub_att', 'end_time_s']

    last_fight_aggs = [pl.col(c).sort_by('fight_date', descending=True).first().alias(f'lf_{c}') for c in agg_cols]
    last_3_aggs     = [pl.col(c).sort_by('fight_date', descending=True).head(3).sum().alias(f'l3_{c}') for c in agg_cols]
    career_aggs     = [pl.col(c).sum().alias(f'ca_{c}') for c in agg_cols]

    def stat_exprs(col_p, alias_p):
        """
        Generates Polars expressions for the 8 derived striking stats from a given window prefix.
        Args:
            col_p: column prefix for the aggregated raw values (e.g. 'lf', 'l3', 'ca')
            alias_p: output column prefix (e.g. 'last_fight', 'last_3', or '')
        Returns:
            list of Polars expressions for the 8 derived stats with safe division
        """
        mins = pl.col(f'{col_p}_end_time_s') / 60.0
        a = f'{alias_p}_' if alias_p else ''
        def safe(expr, alias):
            return pl.when(expr.is_nan() | expr.is_infinite()).then(pl.lit(0.0)).otherwise(expr).alias(alias)
        return [
            safe(pl.col(f'{col_p}_sig_str_landed') / mins,                                         f'{a}slpm'),
            safe(pl.col(f'{col_p}_sig_str_landed') / pl.col(f'{col_p}_sig_str_attempts'),          f'{a}str_acc'),
            safe(pl.col(f'{col_p}_opp_sig_str_landed') / mins,                                     f'{a}sapm'),
            safe(1 - pl.col(f'{col_p}_opp_sig_str_landed') / pl.col(f'{col_p}_opp_sig_str_attempts'), f'{a}str_def'),
            safe(pl.col(f'{col_p}_td_landed') / mins * 15,                                         f'{a}td_avg'),
            safe(pl.col(f'{col_p}_td_landed') / pl.col(f'{col_p}_td_attempts'),                   f'{a}td_acc'),
            safe(1 - pl.col(f'{col_p}_opp_td_landed') / pl.col(f'{col_p}_opp_td_attempts'),       f'{a}td_def'),
            safe(pl.col(f'{col_p}_sub_att') / mins * 15,                                           f'{a}sub_avg'),
        ]

    stats = ['slpm', 'str_acc', 'sapm', 'str_def', 'td_avg', 'td_acc', 'td_def', 'sub_avg']
    trend_exprs = [
        pl.when((pl.col(f'last_3_{s}') / pl.col(s)).is_nan() | (pl.col(f'last_3_{s}') / pl.col(s)).is_infinite())
        .then(pl.lit(1.0))
        .otherwise((pl.col(f'last_3_{s}') / pl.col(s)).clip(0.0, 3.0))
        .alias(f'{s}_trend')
        for s in stats
    ]

    output_cols = (
        ['root_fight_id', 'fighter_role']
        + [f'last_fight_{s}' for s in stats]
        + [f'last_3_{s}' for s in stats]
        + stats
        + [f'{s}_trend' for s in stats]
    )

    return (
        pf.group_by(['root_fight_id', 'fighter_role'])
        .agg(last_fight_aggs + last_3_aggs + career_aggs)
        .with_columns(stat_exprs('lf', 'last_fight') + stat_exprs('l3', 'last_3') + stat_exprs('ca', ''))
        .with_columns(trend_exprs)
        .select(output_cols)
    )


striking = build_striking_stats(prior_fights_df)
print('Striking features shape:', striking.shape)
print('Columns:', striking.columns)
striking.head(2)

## 6. Win/Loss Streak Tracking

Current momentum matters. A fighter on a 5-fight win streak entering a bout carries very different psychological and physical dynamics than one coming off two straight losses.

In [ ]:
def build_streak(prior_fights: pl.DataFrame) -> pl.DataFrame:
    """
    Computes the current consecutive win and loss streaks for each fighter.
    Sorts fights most-recent-first, then counts the unbroken run at the head:
      win_streak  = wins before the first loss in the recency-sorted sequence
      loss_streak = losses before the first win
    Args:
        prior_fights: pl.DataFrame with one row per prior fight including result and fight_date
    Returns:
        pl.DataFrame grouped by (root_fight_id, fighter_role) with win_streak and loss_streak columns
    """
    is_win  = (pl.col('result') == 'win').cast(pl.Int32)
    is_loss = (pl.col('result') == 'loss').cast(pl.Int32)

    pf = (
        prior_fights
        .sort(['root_fight_id', 'fighter_role', 'fight_date'], descending=[False, False, True])
        .with_columns([is_win.alias('is_win'), is_loss.alias('is_loss')])
        .with_columns([
            pl.col('is_loss').cum_sum().over(['root_fight_id', 'fighter_role']).alias('cumsum_loss'),
            pl.col('is_win').cum_sum().over(['root_fight_id', 'fighter_role']).alias('cumsum_win'),
        ])
    )

    return (
        pf.group_by(['root_fight_id', 'fighter_role'])
        .agg([
            pl.col('is_win').filter(pl.col('cumsum_loss') == 0).sum().cast(pl.Int16).alias('win_streak'),
            pl.col('is_loss').filter(pl.col('cumsum_win') == 0).sum().cast(pl.Int16).alias('loss_streak'),
        ])
    )


streak = build_streak(prior_fights_df)
print('Streak features:', streak.shape)
streak.head(5)

## 7. Accumulated Damage

Fighters absorb punishment over their careers. Tracking KO losses and head strikes absorbed helps the model identify fighters whose "chin" may be eroding — a meaningful but invisible pre-fight signal.

In [ ]:
def build_damage_features(prior_fights: pl.DataFrame) -> pl.DataFrame:
    """
    Tracks cumulative punishment absorbed by a fighter across their career.
    Features:
      career_sig_str_absorbed   — total opponent sig strikes landed (chin erosion proxy)
      career_head_str_absorbed  — total opponent head strikes specifically
      career_kd_absorbed        — total knockdowns received
      career_ko_losses          — total KO/TKO losses
      fights_since_last_ko_loss — recency of last KO loss (99 = never)
      ko_losses_last_3          — KO losses among the last 3 fights
    Args:
        prior_fights: pl.DataFrame with one row per prior fight
    Returns:
        pl.DataFrame grouped by (root_fight_id, fighter_role) with 6 damage feature columns
    """
    ko_loss = (pl.col('method') == 'kotko') & (pl.col('result') == 'loss')

    top3_ids = (
        prior_fights
        .with_columns(
            pl.col('fight_date').rank(method='ordinal', descending=True)
            .over(['root_fight_id', 'fighter_role']).alias('fight_rank_desc')
        )
        .filter(pl.col('fight_rank_desc') <= 3)
        .select(['root_fight_id', 'fighter_role', 'prior_fight_id'])
    )
    pf_last3 = prior_fights.join(top3_ids, on=['root_fight_id', 'fighter_role', 'prior_fight_id'], how='inner')

    career_agg = (
        prior_fights.group_by(['root_fight_id', 'fighter_role'])
        .agg([
            pl.col('opp_sig_str_landed').sum().cast(pl.Float32).alias('career_sig_str_absorbed'),
            pl.col('opp_head_landed').sum().cast(pl.Float32).alias('career_head_str_absorbed'),
            pl.col('opp_kd').sum().cast(pl.Float32).alias('career_kd_absorbed'),
            ko_loss.sum().cast(pl.Int16).alias('career_ko_losses'),
        ])
    )

    last3_agg = (
        pf_last3.group_by(['root_fight_id', 'fighter_role'])
        .agg([ko_loss.sum().cast(pl.Int16).alias('ko_losses_last_3')])
    )

    fights_since_ko = (
        prior_fights
        .with_columns([
            ko_loss.alias('is_ko_loss'),
            pl.col('fight_date').rank(method='ordinal', descending=True)
            .over(['root_fight_id', 'fighter_role']).alias('fight_rank_desc'),
        ])
        .group_by(['root_fight_id', 'fighter_role'])
        .agg(pl.col('fight_rank_desc').filter(pl.col('is_ko_loss')).min().alias('last_ko_loss_rank'))
        .with_columns(
            (pl.col('last_ko_loss_rank') - 1).fill_null(99).cast(pl.Int16).alias('fights_since_last_ko_loss')
        )
        .select(['root_fight_id', 'fighter_role', 'fights_since_last_ko_loss'])
    )

    return (
        career_agg
        .join(last3_agg,      on=['root_fight_id', 'fighter_role'], how='left')
        .join(fights_since_ko, on=['root_fight_id', 'fighter_role'], how='left')
        .with_columns([
            pl.col('ko_losses_last_3').fill_null(0),
            pl.col('fights_since_last_ko_loss').fill_null(99),
        ])
    )


damage = build_damage_features(prior_fights_df)
print('Damage features:', damage.shape)
damage.head(3)

## 8. Recency-Weighted Striking Stats

Simple career averages weight a fight from 10 years ago equally to last month. Exponential decay weighting gives the most recent fight ~2x the weight of a fight from 3 fights ago, providing a smoother recency signal than hard windows.

In [ ]:
def build_recency_weighted_stats(prior_fights: pl.DataFrame) -> pl.DataFrame:
    """
    Computes exponential decay-weighted versions of the 8 core striking/grappling stats.
    Weight for a fight N positions back (0 = most recent) = 0.5^(N / HALF_LIFE).
    A half-life of 3 fights means the most recent fight gets ~2x the weight of a fight
    from 3 fights ago, giving a smooth recency signal without truncating history entirely.
    Args:
        prior_fights: pl.DataFrame with one row per prior fight including striking columns
    Returns:
        pl.DataFrame grouped by (root_fight_id, fighter_role) with rw_{stat} columns
        for stat in [slpm, str_acc, sapm, str_def, td_avg, td_acc, td_def, sub_avg]
    """
    HALF_LIFE = 3.0

    end_time_s = (
        pl.col('end_time').str.split(':').list.get(0).cast(pl.Float32) * 60
        + pl.col('end_time').str.split(':').list.get(1).cast(pl.Float32)
    )

    def safe(expr, alias):
        """
        Replaces NaN/Inf with 0.0 in a Polars expression.
        Args:
            expr: Polars expression that may produce NaN or Inf values
            alias: output column name
        Returns:
            Polars expression with NaN/Inf replaced by 0.0
        """
        return pl.when(expr.is_nan() | expr.is_infinite()).then(pl.lit(0.0)).otherwise(expr).alias(alias)

    pf = (
        prior_fights
        .with_columns(end_time_s.alias('end_time_s'))
        .with_columns(
            pl.col('fight_date').rank(method='ordinal', descending=True)
            .over(['root_fight_id', 'fighter_role']).alias('fight_rank_desc')
        )
        .with_columns(
            (pl.lit(0.5) ** ((pl.col('fight_rank_desc') - 1).cast(pl.Float32) / HALF_LIFE)).alias('decay_weight')
        )
    )

    mins = pl.col('end_time_s') / 60.0
    pf = pf.with_columns([
        safe(pl.col('sig_str_landed') / mins,                                   'fight_slpm'),
        safe(pl.col('sig_str_landed') / pl.col('sig_str_attempts'),             'fight_str_acc'),
        safe(pl.col('opp_sig_str_landed') / mins,                               'fight_sapm'),
        safe(1 - pl.col('opp_sig_str_landed') / pl.col('opp_sig_str_attempts'), 'fight_str_def'),
        safe(pl.col('td_landed') / mins * 15,                                   'fight_td_avg'),
        safe(pl.col('td_landed') / pl.col('td_attempts'),                       'fight_td_acc'),
        safe(1 - pl.col('opp_td_landed') / pl.col('opp_td_attempts'),           'fight_td_def'),
        safe(pl.col('sub_att') / mins * 15,                                     'fight_sub_avg'),
    ])

    stats = ['slpm', 'str_acc', 'sapm', 'str_def', 'td_avg', 'td_acc', 'td_def', 'sub_avg']
    weighted_agg = [
        (pl.col(f'fight_{s}') * pl.col('decay_weight')).sum().alias(f'w_{s}_num') for s in stats
    ] + [pl.col('decay_weight').sum().alias('w_denom')]

    output_exprs = [
        safe(pl.col(f'w_{s}_num') / pl.col('w_denom'), f'rw_{s}').cast(pl.Float32) for s in stats
    ]

    return (
        pf.group_by(['root_fight_id', 'fighter_role'])
        .agg(weighted_agg)
        .with_columns(output_exprs)
        .select(['root_fight_id', 'fighter_role'] + [f'rw_{s}' for s in stats])
    )


rw_stats = build_recency_weighted_stats(prior_fights_df)
print('Recency-weighted stats:', rw_stats.shape)
rw_stats.head(3)

## 9. Mirror Augmentation

A subtle but important fix: the UFC dataset assigns the f1/f2 label by a convention that **shifted over time**. In the training set, 56% of fights are labeled "f1 wins", but in the held-out test set only 43% are. Without correction the model learns a spurious directional bias.

The fix: for every training fight, add a mirrored copy where f1 and f2 are swapped and the label is flipped. After augmentation the win rate is exactly 50% by construction, eliminating the bias.

In [ ]:
def mirror_augment(X: np.ndarray, y: np.ndarray, feature_cols: list, rng_seed: int = 42) -> tuple:
    """
    Doubles the training set by adding a f1/f2-swapped copy of every fight to correct label bias.
    The UFC dataset assigns f1/f2 labels by a convention that shifted over time — train has
    56% f1 wins, test only 43%. Without correction the model learns a spurious directional bias.
    For each fight a mirror is created where:
      - f1_* and f2_* columns are swapped
      - *_diff columns are negated (diff = f1 - f2, so flipped diff = -(f1 - f2) = f2 - f1)
      - symmetric context columns (weight_class, fight_format) are unchanged
      - the label is flipped (f1 win -> f2 win)
    After augmentation the training win rate is exactly 50% by construction.
    Args:
        X: feature matrix of shape (n_samples, n_features)
        y: label array of shape (n_samples,) with values 0 or 1
        feature_cols: list of feature column names matching X columns
        rng_seed: random seed for shuffling augmented dataset
    Returns:
        tuple of (X_aug, y_aug) where both arrays are 2x the input size
    """
    col_idx = {col: i for i, col in enumerate(feature_cols)}

    swap_pairs = []
    seen_f2 = set()
    for col in feature_cols:
        if col.startswith('f1_'):
            partner = 'f2_' + col[3:]
            if partner in col_idx and partner not in seen_f2:
                swap_pairs.append((col_idx[col], col_idx[partner]))
                seen_f2.add(partner)

    diff_cols = [i for i, col in enumerate(feature_cols) if col.endswith('_diff')]

    X_mirror = X.copy()
    for i, j in swap_pairs:
        X_mirror[:, i], X_mirror[:, j] = X[:, j].copy(), X[:, i].copy()
    X_mirror[:, diff_cols] = -X[:, diff_cols]

    y_mirror = 1 - y
    X_aug = np.concatenate([X, X_mirror], axis=0)
    y_aug = np.concatenate([y, y_mirror], axis=0)

    rng = np.random.default_rng(rng_seed)
    perm = rng.permutation(len(y_aug))
    return X_aug[perm], y_aug[perm]


# Demonstration with a small synthetic example
print('Mirror augmentation example:')
X_demo = np.array([[1, 2, -1], [3, 4, -2]], dtype=float)
y_demo = np.array([1, 0])
cols_demo = ['f1_age', 'f2_age', 'age_diff']
X_aug, y_aug = mirror_augment(X_demo, y_demo, cols_demo)
print(f'Before: {X_demo.shape}, win rate = {y_demo.mean():.2f}')
print(f'After:  {X_aug.shape},  win rate = {y_aug.mean():.2f} (should be 0.50)')
print('Augmented X:')
print(pd.DataFrame(X_aug, columns=cols_demo))

## 10. Why These Features Were Not Used in the Final Model

This more complex pipeline was explored but the main notebook's simpler `process()` function was used for final results. Key reasons:

1. **Marginal gain**: The additional features (round-level pacing, ELO, recency-weighted stats) added complexity without meaningfully improving ROC-AUC beyond the ~0.63 ceiling.
2. **Polars dependency**: Requires an additional library not needed by the main notebook.
3. **Round data sparsity**: Many fighters have missing round-level data, requiring extra handling.
4. **Interpretability**: The main notebook's 309-feature set was already rich enough for SHAP analysis.

Future work could revisit these features with a more sophisticated model or additional data cleaning.